# Revisione del filtro: tenute / scartate
Legge `kept.csv` e `discarded.csv` prodotti da `pixi run filter`. Esegui tutte le celle: nessun widget, i parametri sono costanti nella prima cella.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display

OUT = Path("../output/filtered")       # cartella con kept.csv / discarded.csv
RESULTS = Path("../output/results.csv")  # risultati età (facoltativo, per i dettagli per volto)
THUMB = (200, 120)                      # miniatura (larghezza, altezza)
COLS = 6                                # miniature per riga
N = 60                                  # immagini mostrate per gruppo (None = tutte)
SEED = 0                                # campionamento casuale riproducibile

kept = pd.read_csv(OUT / "kept.csv")
disc = pd.read_csv(OUT / "discarded.csv")
print(f"tenute {len(kept)} | scartate {len(disc)} ({len(disc) / max(1, len(kept) + len(disc)):.0%})")
display(disc.reason.value_counts().to_frame("n"))

## Distribuzione delle metriche
Serve a scegliere le soglie: `color_dev_frac_in` (tenute vs scartate) e la probabilità di minorenne.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, col, title in [(axs[0], "color_dev_frac_in", "color_dev_frac_in"), (axs[1], "p_minor_max", "p_minor massima per immagine")]:
    if col not in kept: ax.set_visible(False); continue
    both = pd.concat([kept[col], disc[col]]).dropna()
    bins = np.linspace(0, max(both.max(), 1e-6), 30)
    ax.hist(kept[col].dropna(), bins=bins, alpha=.6, label="tenute")
    ax.hist(disc[col].dropna(), bins=bins, alpha=.6, label="scartate")
    ax.set_yscale("log"); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()
display(pd.concat([kept.assign(gruppo="tenute"), disc.assign(gruppo="scartate")]).groupby("gruppo")[["color_dev_frac_in"]].describe())

## Mosaici
Ogni miniatura riporta in alto nome del frame, `color_dev_frac_in` e `p_minor`. Le scartate sono raggruppate per motivo, a campione casuale.

In [ ]:
def thumb(r):
    try:
        im = Image.open(r.image_path).convert("RGB").resize(THUMB)
    except OSError:
        im = Image.new("RGB", THUMB, (60, 60, 60))
    d = ImageDraw.Draw(im)
    pm = getattr(r, "p_minor_max", float("nan"))
    txt = f"{Path(r.file).name[6:12]} {Path(r.file).name.split('_')[1][5:]} cd={r.color_dev_frac_in:.2f} pm={pm:.2f}"
    d.rectangle([0, 0, THUMB[0], 13], fill=(0, 0, 0)); d.text((3, 1), txt, fill=(255, 255, 255))
    return im

def mosaic(df):
    df = df if N is None or len(df) <= N else df.sample(N, random_state=SEED)
    tiles = [thumb(r) for r in df.itertuples()]
    rows = -(-len(tiles) // COLS)
    canvas = Image.new("RGB", (COLS * THUMB[0], rows * THUMB[1]))
    for i, t in enumerate(tiles):
        canvas.paste(t, ((i % COLS) * THUMB[0], (i // COLS) * THUMB[1]))
    return canvas

for reason, g in disc.groupby("reason"):
    print(f"=== scartate: {reason} ({len(g)}) ===")
    display(mosaic(g))
print(f"=== tenute ({len(kept)}) ===")
display(mosaic(kept))

## Ispezione di un singolo frame
Imposta `FRAME` (parte del nome file) per vedere tutte le righe di quel frame, con la decisione e i dettagli dei volti da `results.csv`.

In [ ]:
FRAME = "frame_000755"

allr = pd.concat([kept.assign(stato="tenuta"), disc.assign(stato="scartata")])
sel = allr[allr.file.str.contains(FRAME)]
display(sel[["file", "stato", "reason", "n_est", "minor", "color_dev_frac_in"] + (["p_minor_max"] if "p_minor_max" in sel else [])])
if RESULTS.exists():
    res = pd.read_csv(RESULTS)
    res = res[res.file.map(lambda p: Path(p).name).isin(sel.file.map(lambda p: Path(p).name))]
    display(res[[c for c in ["file", "age_bin", "age_conf", "p_minor", "det_score"] if c in res]])
n = len(sel)
if n:
    fig, axs = plt.subplots(1, n, figsize=(4.5 * n, 3), squeeze=False)
    for ax, r in zip(axs[0], sel.itertuples()):
        ax.imshow(Image.open(r.image_path)); ax.axis("off")
        ax.set_title(f"{Path(r.file).name.split('_')[1]} - {r.stato}" + (f"\n{r.reason}" if isinstance(r.reason, str) else ""), fontsize=8)
    plt.tight_layout(); plt.show()